# AI-Based Industrial Machine Health & Failure Prediction
## Machine Learning Capstone Project (23CSE301) — Review 1 Submission
**Academic Year:** 2026–2027 | **Review Scope:** Review 1 Phase Only (Total: 25 Marks)  
**Evaluator Note:** All cells are written in simple, modular, evaluator-friendly Python with clear explanations and zero hidden state.


## 1. Problem Statement
Modern industrial manufacturing relies on computerized numerical control (CNC) machines, milling equipment, and automated rotating assembly units. Unexpected mechanical or electrical machine breakdown causes expensive production downtime, damaged workpieces, and occupational safety hazards.

Industrial machinery experiences operating degradation driven by physical factors including thermal friction (`Air` and `Process Temperatures`), mechanical stress (`Rotational speed [rpm]` and `Torque [Nm]`), and cutting tool fatigue (`Tool wear [min]`).

The objective of this project is to build an end-to-end Machine Learning pipeline for predictive maintenance using the **AI4I 2020 Predictive Maintenance Dataset**:
1. **Regression Track**: Predict continuous tool wear accumulation (`Tool wear [min]`) to monitor equipment aging and forecast tool replacement schedules.
2. **Classification Track (Part A)**: Predict machine operational failure (`Machine failure`, binary 0/1) using baseline statistical and machine learning classifiers to trigger automated protective shutdowns.


## 2. Dataset Description
The dataset contains **10,000 operational records** with 14 variables representing synthetic but physically realistic milling machine operations:
- `UDI`: Row identifier (1 to 10,000) [Excluded from model features to prevent arbitrary index memorization].
- `Product ID`: Variant identifier consisting of quality letter (L, M, H) and serial number [Excluded from model features].
- `Type`: Machine product quality variant:
  - **L (Low)**: 50% of units (product variants with standard tolerances)
  - **M (Medium)**: 30% of units
  - **H (High)**: 20% of units (heavy-duty precision variants)
- `Air temperature [K]`: Generated using a random walk normalized to standard deviations around ~300 K.
- `Process temperature [K]`: Generated as air temperature + ~10 K thermal offset due to machine friction.
- `Rotational speed [rpm]`: Spindle speed calculated around an operating power band.
- `Torque [Nm]`: Resistance torque normally distributed around ~40 Nm.
- `Tool wear [min]`: Cumulative active cutting time before replacement.
- `Machine failure`: Ground-truth binary target (0 = Normal Operation, 1 = Machine Failure).
- `TWF`, `HDF`, `PWF`, `OSF`, `RNF`: Five specific failure modes (Tool Wear Failure, Heat Dissipation Failure, Power Failure, Overstrain Failure, Random Network Failure). *Audited below for target leakage.*


## 3. Import Libraries
We import standard scientific and machine learning libraries. All seeds are fixed to `42` for exact reproducibility.


In [ ]:
# Core scientific packages
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Model Selection
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA

# 10 Regression Algorithms
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

# 5 Classification Algorithms (Part A)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Configure plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11

print("All required libraries successfully loaded!")


## 4. Load Dataset
We load `ai4i2020.csv` directly from the local data directory and preview initial observations.


In [ ]:
# Load dataset
data_path = 'ai4i2020.csv' if os.path.exists('ai4i2020.csv') else 'data/ai4i2020.csv'
df = pd.read_csv(data_path)

print("Dataset successfully loaded.")
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 5. Dataset Audit (Review 1 Rubric A1)
A comprehensive audit of dataset shape, data types, missing-value counts, duplicates, and target distributions.


In [ ]:
# Dataset Audit Report
print("=" * 60)
print("              DATASET AUDIT REPORT (Review 1 A1)            ")
print("=" * 60)
print(f"Dataset Shape: Rows = {df.shape[0]}, Columns = {df.shape[1]}")
print(f"Total Duplicate Rows: {df.duplicated().sum()}")
print("
--- Missing Value Counts per Column ---")
print(df.isnull().sum())

print("
--- Data Types & Non-Null Summary ---")
print(df.dtypes)

print("
--- Target Variable Distribution: Machine failure ---")
failure_counts = df['Machine failure'].value_counts()
failure_percentages = df['Machine failure'].value_counts(normalize=True) * 100
audit_target_df = pd.DataFrame({
    'Count': failure_counts,
    'Percentage (%)': failure_percentages.round(2)
})
print(audit_target_df)

print("
--- 5-Point Summary for Numerical Sensor Features ---")
sensor_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
print(df[sensor_cols].describe().T[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(2))


### Dataset Audit Commentary
- **Shape & Completeness**: The dataset contains exactly 10,000 instances and 14 features with **0 missing values** and **0 duplicate entries**.
- **Data Types**: Contains 2 string columns (`Product ID`, `Type`), 3 float columns (`Air temperature`, `Process temperature`, `Torque`), and 9 integer columns (`UDI`, `Rotational speed`, `Tool wear`, `Machine failure`, and 5 failure flags).
- **Target Distribution**: Exactly 9,661 instances (96.61%) are non-failure cycles and 339 instances (3.39%) are machine failures. This confirms significant class imbalance (~28.5:1 ratio), which necessitates stratified splitting and threshold-independent metrics.
